# Stage 3 Wander — Boundary and Obstacle Avoidance

Robot drives forward (`WANDER`) and reacts to two hazards:
- **Yellow tape** in the bottom ROI → `AVOID_BOUNDARY`
- **Canny-edge density** in the front ROI → `AVOID_OBSTACLE`

No cap detection, no arm control, no Roboflow.

States: `WANDER` → `AVOID_BOUNDARY` / `AVOID_OBSTACLE` → `WANDER` → …

## 1. Imports

In [ ]:
import cv2
import numpy as np
import time
import ipywidgets as widgets
from IPython.display import display
from jetbot import Camera, Robot, bgr8_to_jpeg

print('Imports OK')

## 2. Configuration

Tune `YELLOW_BOUNDARY_THRESHOLD` and `OBSTACLE_EDGE_THRESHOLD` on-site  
using **Test 5** before running the full wander.

In [ ]:
# ── Camera ──────────────────────────────────────────────────────────
CAMERA_WIDTH  = 300
CAMERA_HEIGHT = 300

# ── Motor speeds (sign applied in logic) ────────────────────────────
FORWARD_SPEED = 0.18   # normal wander speed
REVERSE_SPEED = 0.15   # backing away from a hazard
TURN_SPEED    = 0.22   # in-place turn during avoidance
MAX_SPEED     = 0.25   # hard clamp on all motor commands

# ── Avoidance timing ────────────────────────────────────────────────
AVOID_REVERSE_TIME = 0.40   # seconds: phase 1 reverse
AVOID_TURN_TIME    = 0.55   # seconds: phase 2 turn
WANDER_COOLDOWN    = 0.50   # seconds: free drive after avoidance

# ── ROI fractions (of frame height) ─────────────────────────────────
BOTTOM_ROI_FRACTION   = 0.30   # bottom 30 % for yellow tape
FRONT_ROI_TOP_FRAC    = 0.15   # obstacle zone starts at 15 %
FRONT_ROI_BOTTOM_FRAC = 0.65   # obstacle zone ends   at 65 %

# ── Detection thresholds ────────────────────────────────────────────
YELLOW_BOUNDARY_THRESHOLD = 1800   # yellow pixels to trigger boundary
OBSTACLE_EDGE_THRESHOLD   = 500    # edge   pixels to trigger obstacle

# ── HSV range for yellow floor tape ─────────────────────────────────
TAPE_HSV_LOWER = np.array([15,  60,  60])
TAPE_HSV_UPPER = np.array([45, 255, 255])

# ── Canny parameters ────────────────────────────────────────────────
CANNY_LOW  = 40
CANNY_HIGH = 120

print('Configuration loaded.')
print('  Speeds — forward:', FORWARD_SPEED,
      ' turn:', TURN_SPEED,
      ' reverse:', REVERSE_SPEED)
print('  Thresholds — yellow:', YELLOW_BOUNDARY_THRESHOLD,
      ' obstacle:', OBSTACLE_EDGE_THRESHOLD)

## 3. Camera and Robot

Camera is a singleton — safe to re-run if it already exists.

In [ ]:
camera = Camera.instance(width=CAMERA_WIDTH, height=CAMERA_HEIGHT)

image_widget = widgets.Image(format='jpeg', width=CAMERA_WIDTH,  height=CAMERA_HEIGHT)
mask_widget  = widgets.Image(format='jpeg', width=CAMERA_WIDTH,  height=CAMERA_HEIGHT)
state_label  = widgets.Label(value='State: STOPPED')
motor_label  = widgets.Label(value='Motors: L=0.00  R=0.00')
yellow_label = widgets.Label(value='Yellow: 0  (L=0 R=0)')
obs_label    = widgets.Label(value='Edges: 0  (L=0 R=0)')
turn_label   = widgets.Label(value='Turn dir: -')

display(widgets.VBox([
    widgets.HBox([image_widget, mask_widget]),
    widgets.VBox([state_label, motor_label,
                  yellow_label, obs_label, turn_label])
]))
print('Camera ready.')

In [ ]:
robot = Robot()
robot.stop()
print('Robot initialized and stopped.')

## 4. Helper Functions

In [ ]:
def clamp(val, lo, hi):
    if val < lo: return lo
    if val > hi: return hi
    return val


def safe_stop():
    try:
        robot.left_motor.value  = 0.0
        robot.right_motor.value = 0.0
    except Exception:
        pass


def set_drive(left, right):
    robot.left_motor.value  = clamp(left,  -MAX_SPEED, MAX_SPEED)
    robot.right_motor.value = clamp(right, -MAX_SPEED, MAX_SPEED)


print('clamp / safe_stop / set_drive defined.')

## 5. Yellow Boundary Detection

Inspects the **bottom 30 %** of the frame.  
Returns yellow pixel counts for the full ROI, left half, and right half  
so avoidance can choose which way to turn.

In [ ]:
def detect_yellow_boundary(frame):
    h         = frame.shape[0]
    roi_start = int(h * (1.0 - BOTTOM_ROI_FRACTION))
    roi       = frame[roi_start:, :]

    hsv_roi  = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
    roi_mask = cv2.inRange(hsv_roi, TAPE_HSV_LOWER, TAPE_HSV_UPPER)

    yellow_area  = int(np.sum(roi_mask > 0))
    mid          = roi_mask.shape[1] // 2
    yellow_left  = int(np.sum(roi_mask[:, :mid] > 0))
    yellow_right = int(np.sum(roi_mask[:, mid:] > 0))

    full_mask = np.zeros(frame.shape[:2], dtype=np.uint8)
    full_mask[roi_start:, :] = roi_mask

    return {
        'detected':     yellow_area >= YELLOW_BOUNDARY_THRESHOLD,
        'yellow_area':  yellow_area,
        'yellow_left':  yellow_left,
        'yellow_right': yellow_right,
        'yellow_mask':  full_mask,
    }


print('detect_yellow_boundary() defined.')

## 6. Obstacle Detection

Inspects the **front 15 – 65 %** of the frame (middle strip).  
Yellow pixels are masked out first so floor tape does not count as an obstacle.  
Canny edges are counted; high density → something is in the path.

In [ ]:
def detect_obstacle(frame):
    h, w    = frame.shape[:2]
    roi_top = int(h * FRONT_ROI_TOP_FRAC)
    roi_bot = int(h * FRONT_ROI_BOTTOM_FRAC)
    roi     = frame[roi_top:roi_bot, :]

    # Mask yellow pixels so floor tape does not register as an obstacle
    hsv_roi     = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
    yellow_mask = cv2.inRange(hsv_roi, TAPE_HSV_LOWER, TAPE_HSV_UPPER)

    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    gray[yellow_mask > 0] = 0

    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    edges   = cv2.Canny(blurred, CANNY_LOW, CANNY_HIGH)
    edges   = cv2.dilate(edges, None, iterations=1)

    mid        = edges.shape[1] // 2
    edge_left  = int(np.sum(edges[:, :mid] > 0))
    edge_right = int(np.sum(edges[:, mid:] > 0))
    edge_total = edge_left + edge_right

    full_mask = np.zeros(frame.shape[:2], dtype=np.uint8)
    full_mask[roi_top:roi_bot, :] = edges

    return {
        'detected':   edge_total >= OBSTACLE_EDGE_THRESHOLD,
        'edge_total': edge_total,
        'edge_left':  edge_left,
        'edge_right': edge_right,
        'mask':       full_mask,
    }


print('detect_obstacle() defined.')

## 7. Debug Visualization

Left widget — annotated live frame:
- **Cyan rectangle** (bottom): yellow tape detection zone
- **Orange rectangle** (middle): obstacle detection zone
- Rectangle turns **red** when that hazard fires

Right widget — detection mask:
- **Cyan** pixels = yellow tape
- **Orange** pixels = obstacle edges

In [ ]:
def draw_debug(frame, boundary, obstacle, state, left_spd, right_spd):
    out  = frame.copy()
    h, w = out.shape[:2]

    # Yellow ROI
    roi_y   = int(h * (1.0 - BOTTOM_ROI_FRACTION))
    y_color = (0, 0, 255) if boundary['detected'] else (0, 220, 220)
    cv2.rectangle(out, (0, roi_y), (w - 1, h - 1), y_color, 2)

    # Obstacle ROI
    obs_top = int(h * FRONT_ROI_TOP_FRAC)
    obs_bot = int(h * FRONT_ROI_BOTTOM_FRAC)
    o_color = (0, 0, 255) if obstacle['detected'] else (255, 130, 0)
    cv2.rectangle(out, (0, obs_top), (w - 1, obs_bot), o_color, 2)

    # Vertical centerline
    cv2.line(out, (w // 2, 0), (w // 2, h), (70, 70, 70), 1)

    # Text overlay
    cv2.putText(out, 'S:' + state,
                (5, 18), cv2.FONT_HERSHEY_SIMPLEX, 0.50, (255, 255, 255), 1)
    cv2.putText(out,
                'L=' + str(round(left_spd, 2)) + ' R=' + str(round(right_spd, 2)),
                (5, 34), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (255, 255, 255), 1)
    cv2.putText(out, 'Y=' + str(boundary['yellow_area']),
                (5, 49), cv2.FONT_HERSHEY_SIMPLEX, 0.38, (0, 240, 240), 1)
    cv2.putText(out, 'E=' + str(obstacle['edge_total']),
                (5, 62), cv2.FONT_HERSHEY_SIMPLEX, 0.38, (255, 130, 0), 1)

    if boundary['detected']:
        cv2.putText(out, 'BOUNDARY',
                    (5, 84), cv2.FONT_HERSHEY_SIMPLEX, 0.60, (0, 0, 255), 2)
    elif obstacle['detected']:
        cv2.putText(out, 'OBSTACLE',
                    (5, 84), cv2.FONT_HERSHEY_SIMPLEX, 0.60, (0, 80, 255), 2)

    return out


print('draw_debug() defined.')

## 8. State Machine

| State | Behaviour |
|-------|-----------|
| `WANDER` | Drive forward. Check both hazards every frame. |
| `AVOID_BOUNDARY` | Phase 1: reverse. Phase 2: turn. Phase 3: resume `WANDER`. |
| `AVOID_OBSTACLE` | Same three-phase sequence as `AVOID_BOUNDARY`. |
| `STOPPED` | Emergency stop. No auto-recovery. |

**Priority:** boundary > obstacle > wander.  
**Turn direction** = away from the half of the ROI that has more signal.  
**Cooldown:** 0.5 s of free driving after each avoidance before re-checking.

In [ ]:
nav_state           = 'WANDER'
avoid_start_time    = 0.0
avoid_turn_right    = True
wander_cooldown_end = 0.0
last_left_spd       = 0.0
last_right_spd      = 0.0

print('State machine ready.  nav_state =', nav_state)

In [ ]:
def execute(change):
    global nav_state, avoid_start_time, avoid_turn_right
    global wander_cooldown_end, last_left_spd, last_right_spd

    frame     = change['new']
    left_spd  = 0.0
    right_spd = 0.0

    try:
        boundary = detect_yellow_boundary(frame)
        obstacle = detect_obstacle(frame)
        now      = time.time()

        # ── State machine ────────────────────────────────────────

        if nav_state == 'STOPPED':
            pass   # only manual reset can leave STOPPED

        elif nav_state in ('AVOID_BOUNDARY', 'AVOID_OBSTACLE'):
            elapsed = now - avoid_start_time
            if elapsed < AVOID_REVERSE_TIME:
                # Phase 1: reverse straight back
                left_spd  = -REVERSE_SPEED
                right_spd = -REVERSE_SPEED
            elif elapsed < AVOID_REVERSE_TIME + AVOID_TURN_TIME:
                # Phase 2: turn away from the hazard side
                if avoid_turn_right:
                    left_spd  =  TURN_SPEED
                    right_spd = -TURN_SPEED
                else:
                    left_spd  = -TURN_SPEED
                    right_spd =  TURN_SPEED
            else:
                # Phase 3: avoidance done, resume wander with cooldown
                wander_cooldown_end = now + WANDER_COOLDOWN
                nav_state = 'WANDER'
                left_spd  = FORWARD_SPEED
                right_spd = FORWARD_SPEED

        elif nav_state == 'WANDER':
            if now < wander_cooldown_end:
                # Post-avoidance free run — skip detection
                left_spd  = FORWARD_SPEED
                right_spd = FORWARD_SPEED
            elif boundary['detected']:
                # Boundary has higher priority than obstacle
                avoid_turn_right = boundary['yellow_left'] >= boundary['yellow_right']
                avoid_start_time = now
                nav_state        = 'AVOID_BOUNDARY'
                left_spd         = -REVERSE_SPEED
                right_spd        = -REVERSE_SPEED
            elif obstacle['detected']:
                avoid_turn_right = obstacle['edge_left'] >= obstacle['edge_right']
                avoid_start_time = now
                nav_state        = 'AVOID_OBSTACLE'
                left_spd         = -REVERSE_SPEED
                right_spd        = -REVERSE_SPEED
            else:
                left_spd  = FORWARD_SPEED
                right_spd = FORWARD_SPEED

        # ── Drive ────────────────────────────────────────────────
        set_drive(left_spd, right_spd)
        last_left_spd  = left_spd
        last_right_spd = right_spd

        # ── Debug display ────────────────────────────────────────
        annotated          = draw_debug(frame, boundary, obstacle,
                                        nav_state, left_spd, right_spd)
        image_widget.value = bgr8_to_jpeg(annotated)

        # Cyan = yellow tape pixels, orange = obstacle edge pixels
        combined = np.zeros((CAMERA_HEIGHT, CAMERA_WIDTH, 3), dtype=np.uint8)
        ym = boundary['yellow_mask']
        om = obstacle['mask']
        combined[ym > 0] = [0, 255, 255]
        combined[om > 0] = [0, 140, 255]
        mask_widget.value = bgr8_to_jpeg(combined)

        state_label.value  = 'State: ' + nav_state
        motor_label.value  = ('Motors L=' + str(round(left_spd, 3)) +
                              '  R=' + str(round(right_spd, 3)))
        yellow_label.value = ('Yellow: ' + str(boundary['yellow_area']) +
                              '  L=' + str(boundary['yellow_left']) +
                              '  R=' + str(boundary['yellow_right']))
        obs_label.value    = ('Edges: ' + str(obstacle['edge_total']) +
                              '  L=' + str(obstacle['edge_left']) +
                              '  R=' + str(obstacle['edge_right']))
        td = 'RIGHT' if avoid_turn_right else 'LEFT'
        turn_label.value   = 'Turn dir: ' + td

    except Exception as e:
        safe_stop()
        print('[ERROR] ' + str(e))
        raise


print('execute() defined.')

## 9. Start Navigation

Run this cell to attach the callback.  
**The robot will start moving immediately.**

In [ ]:
nav_state           = 'WANDER'
avoid_start_time    = 0.0
avoid_turn_right    = True
wander_cooldown_end = 0.0
last_left_spd       = 0.0
last_right_spd      = 0.0

camera.unobserve_all()
camera.observe(execute, names='value')
print('Navigation started.  State:', nav_state)

## 10. Emergency Stop

Run immediately if the robot misbehaves.

In [ ]:
camera.unobserve_all()
safe_stop()
nav_state = 'STOPPED'
state_label.value = 'State: STOPPED'
motor_label.value = 'Motors: L=0.00  R=0.00'
print('EMERGENCY STOP — camera detached, motors zeroed.')

## 11. Reset and Restart

After stopping, run this instead of re-running all cells.

In [ ]:
camera.unobserve_all()
safe_stop()

nav_state           = 'WANDER'
avoid_start_time    = 0.0
avoid_turn_right    = True
wander_cooldown_end = 0.0
last_left_spd       = 0.0
last_right_spd      = 0.0

camera.observe(execute, names='value')
print('Reset to WANDER.  Navigation restarted.')

## 12. Test Procedures

Run these **before** starting the full wander to verify each subsystem.

---

### Test 1 — Yellow Boundary Detection (static)

Hold yellow tape under the camera or position the robot near the arena edge.

```python
frame  = camera.value.copy()
result = detect_yellow_boundary(frame)
print('detected:', result['detected'],
      ' area:', result['yellow_area'],
      ' L:', result['yellow_left'],
      ' R:', result['yellow_right'])

vis = frame.copy()
h   = vis.shape[0]
roi_y = int(h * (1.0 - BOTTOM_ROI_FRACTION))
cv2.rectangle(vis, (0, roi_y), (vis.shape[1]-1, h-1), (0, 220, 220), 2)

mask_bgr = cv2.cvtColor(result['yellow_mask'], cv2.COLOR_GRAY2BGR)
display(widgets.Image(value=bgr8_to_jpeg(vis),      format='jpeg'))
display(widgets.Image(value=bgr8_to_jpeg(mask_bgr), format='jpeg'))
```

**Pass:** `detected` is `True` with tape present, `False` on plain floor.  
Adjust `YELLOW_BOUNDARY_THRESHOLD` (lower = more sensitive).

---

### Test 2 — Obstacle Detection (static)

Hold a cardboard box ~30 cm in front of the robot.

```python
frame  = camera.value.copy()
result = detect_obstacle(frame)
print('detected:', result['detected'],
      ' edges:', result['edge_total'],
      ' L:', result['edge_left'],
      ' R:', result['edge_right'])

edge_bgr = cv2.cvtColor(result['mask'], cv2.COLOR_GRAY2BGR)
display(widgets.Image(value=bgr8_to_jpeg(edge_bgr), format='jpeg'))
```

**Pass:** `detected` is `True` with box present, `False` on clear floor.  
Adjust `OBSTACLE_EDGE_THRESHOLD` or `CANNY_LOW/HIGH` if the floor triggers false positives.

---

### Test 3 — Motor Directions (robot on blocks)

Lift the robot so tracks spin freely.

```python
import time
print('Forward 1 s')
set_drive(FORWARD_SPEED,  FORWARD_SPEED);  time.sleep(1.0); safe_stop()
time.sleep(0.5)
print('Reverse 1 s')
set_drive(-REVERSE_SPEED, -REVERSE_SPEED); time.sleep(1.0); safe_stop()
time.sleep(0.5)
print('Turn right 1 s')
set_drive(TURN_SPEED,     -TURN_SPEED);    time.sleep(1.0); safe_stop()
```

**Pass:** both tracks spin in the correct direction.

---

### Test 4 — Full Wander (place robot in arena)

Run **Section 9 — Start Navigation**.  
Watch the live feed and the state label.

**Expected:**
1. Robot drives forward in `WANDER`.
2. Yellow tape in the bottom ROI → `AVOID_BOUNDARY`: reverses, turns, returns to `WANDER`.
3. Box in the front ROI → `AVOID_OBSTACLE`: same three-phase sequence.
4. Cycle continues indefinitely.

If the robot turns the **wrong way**, swap the `>=` to `<` in the `avoid_turn_right` assignment  
inside the relevant branch of `execute()`.

---

### Test 5 — Threshold Calibration (robot stationary, clear arena)

```python
frame = camera.value.copy()
b = detect_yellow_boundary(frame)
o = detect_obstacle(frame)
print('Baseline — yellow:', b['yellow_area'], '  edges:', o['edge_total'])
```

Set `YELLOW_BOUNDARY_THRESHOLD` ≈ **3× baseline yellow**.  
Set `OBSTACLE_EDGE_THRESHOLD`   ≈ **3× baseline edge count**.